# Stockfish players

In [ ]:
import sys
sys.path.append("..")

from local_evaluation import (
    StockfishAgent, 
    evaluate_against_opponent, 
)
import pandas as pd
import chess 

In [ ]:
num_games = 500

In [ ]:
# Run evaluation against each opponent using ThreadPoolExecutor
def evaluate_opponent_task(opponent_name, opponent_agent, player_agent):
    """Task wrapper for parallel evaluation."""

    try:
        result = evaluate_against_opponent(
            player_agent=player_agent,
            opponent_name=opponent_name,
            opponent_agent=opponent_agent,
            num_games=num_games,
            verbose=False,
            debug=False,
        )
        return result
    except Exception as e:
        print(f"\nError evaluating against {opponent_name}: {e}")
        import traceback
        traceback.print_exc()
        return None
    finally:
        # Clean up Stockfish agents (the original one, game-specific ones are cleaned up in evaluate_against_opponent)
        if hasattr(opponent_agent, 'close'):
            opponent_agent.close()

In [ ]:
opponent_name = "Stockfish (depth 1, skill 0)"
opponent_agent = StockfishAgent(
    depth=1, 
    skill_level=0, 
    time_limit_ms=1000,
)
result = evaluate_opponent_task(
    opponent_name,
    opponent_agent,
    player_agent={
        "type": "stockfish",
        "depth": 10,
        "skill_level": 20,
        "time_limit_ms": 1000
    }
)

In [ ]:
def get_player_win(row):
    result = row["result"].split(" ")[0].lower()
    player_color = row["player_color"]
    return player_color == result

def get_player_score(row):
    player_color = row["player_color"]
    if player_color == "black":
        return row["black_acpl"]
    elif player_color == "white":
        return row["white_acpl"]
    else:
        print("Error")
game_df = pd.DataFrame(result.games)
game_df["player_win"] = game_df.apply(get_player_win, axis=1)
game_df["player_score"] = game_df.apply(get_player_score, axis=1)
game_df["illegal_move"] = game_df["result"].apply(lambda x: "resigned" in x)

In [ ]:
game_df.to_parquet("game.parquet")

In [ ]:
game_df.groupby("player_color").agg(  
    avg_player_win=("player_win", "mean"),  
    avg_player_score=("player_score", "mean"),
    avg_illegal_move=("illegal_move", "mean"),
)

In [ ]:
board_list = []
game_ids = []
for game_id, row in game_df.iterrows():  
    board = chess.Board()  
    moves = row["move_history"]  
  
    is_white = row["player_color"] == "white"  
  
    for i, move in enumerate(moves):  
        board.push_uci(move)  
  
        if (is_white and i % 2 == 0) or (not is_white and i % 2 == 1):  
            board_list.append(board.fen())  
            game_ids.append(game_id)  

In [ ]:
board_df = pd.DataFrame({"board": board_list, "game_id": game_ids})

In [ ]:
len(board_df)

In [ ]:
# True if a board appears more than once in the dataset  
is_duplicated_board = board_df["board"].duplicated(keep=False)  
  
# For each game: are ALL its boards duplicated?  
games_all_duplicated = (  
    board_df.assign(is_dup=is_duplicated_board)  
            .groupby("game_id")["is_dup"]  
            .all()  
)  
  
percentage = games_all_duplicated.mean() * 100  
print(f"Prob. to play same every moves {percentage}%")

# Label

In [ ]:
import chess  

def render_board_unicode(board) -> str:
    UNICODE_PIECES = {
        'P': '♙',  # White pawn
        'R': '♖',  # White rook
        'N': '♘',  # White knight
        'B': '♗',  # White bishop
        'Q': '♕',  # White queen
        'K': '♔',  # White king
        
        'p': '♟',  # Black pawn
        'r': '♜',  # Black rook
        'n': '♞',  # Black knight
        'b': '♝',  # Black bishop
        'q': '♛',  # Black queen
        'k': '♚',  # Black king
    }

    lines = []

    # Board coordinates
    files = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
    ranks = ['8', '7', '6', '5', '4', '3', '2', '1']

    # Add top coordinate line with proper alignment
    # Each square is 3 characters wide, so we need to center each letter
    coord_parts = []
    for file in files:
        coord_parts.append(f" {file} ")  # 3-character spacing to match board squares
    coord_line = "   " + "".join(coord_parts) + "  "
    lines.append(coord_line)
    # Calculate border width: 8 squares × 3 characters each = 24 characters
    border_width = len(files) * 3
    lines.append("   +" + "-" * border_width + "+")

    # Render board squares
    for rank_idx, rank in enumerate(ranks):
        line_parts = []
        
        # Add rank coordinate
        line_parts.append(f"{rank} |")
        
        # Add squares
        for file_idx, file in enumerate(files):
            square = chess.parse_square(file + rank)
            piece = board.piece_at(square)
            
            # Get piece symbol or empty square character
            if piece is None:
                piece_char = "·"  # Empty square
            else:
                piece_char = UNICODE_PIECES[piece.symbol()]
            
            # Format square
            square_str = f" {piece_char} "
            line_parts.append(square_str)
        
        # Add closing coordinate
        line_parts.append(f"| {rank}")
        lines.append("".join(line_parts))

    # Add bottom coordinate line
    lines.append("   +" + "-" * border_width + "+")
    coord_line = "   " + "".join(coord_parts) + "  "
    lines.append(coord_line)

    return "\n".join(lines)

In [ ]:
SYSTEM_PROMPT = """  
You are a chess annotation engine generating SIMPLE, STRUCTURED reasoning for a SMALL language model.  

## Guidelines:  
- First, state the game phase (early game / mid game / end game) in 1 sentence.  
- Describe one or two candidate moves, including the target move, using one to two short sentences.
  Start with “As White, …” or “As Black, …”
  If there is a bad move (e.g., capturing a pawn but losing the queen), mention it and clearly reject it with a reason.
- Check if one or two candidate moves are legal moves or not in 1 sentence.
- Finally, conclude by selecting the target move with a brief supporting reason in exactly 1 sentence. 

## Note:
- Do not overcomplicate the reasoning.  
- Keep explanations consistent and easy to understand.  
- Do not exceed 100 words total. 

## Context
Your side: {side_to_move}
Target move: {target_move}
Legal moves: {legal_moves_uci_list}
Board position:  
{board_utf}
"""

In [ ]:
import pandas as pd
import chess

model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"

In [ ]:
df = pd.concat(
    [
        pd.read_parquet("game-500-depth0.parquet"),
        pd.read_parquet("game-100-depth0.parquet"),
    ], ignore_index=True
)

# df = df[:1]
df.shape

In [ ]:
train_data = []

for _,row in df.iterrows():
    is_white = row["player_color"] == "white"
    board = chess.Board()
        
    for i, move in enumerate(row["move_history"]):  
        if (is_white and i % 2 == 0) or (not is_white and i % 2 == 1):
            side_to_move = "White" if is_white else "Black"
            legal_moves_uci_list = [m.uci() for m in board.legal_moves]
            board_utf = render_board_unicode(board)
            ## train data
            data = {
                "board_utf": board_utf,
                "board_fen": board.fen(),
                "legal_moves_uci_list": legal_moves_uci_list,
                "side_to_move": side_to_move,
                "target_move": move,
            }	
            train_data.append(data)
        board.push_uci(move)
        
train_df = pd.DataFrame(train_data)
print(train_df.shape)
train_df = train_df.drop_duplicates(subset=["board_utf"]).reset_index(drop=True)
train_df.shape

In [ ]:
messages = []
for _,row in train_df.iterrows():
    ## prompt
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        target_move=row["target_move"],
        legal_moves_uci_list=row["legal_moves_uci_list"],
        board_utf=row["board_utf"],
    )
    m = [{"role": "user", "content": prompt}]  
    messages.append(m)
len(messages)

In [ ]:
from transformers import AutoTokenizer  
tokenizer = AutoTokenizer.from_pretrained(model_name)  
texts = []
for m in messages:
    text = tokenizer.apply_chat_template(  
        m,  
        tokenize=False,  
        add_generation_prompt=True
    )
    texts.append(text)
len(texts)

In [ ]:
from vllm import LLM, SamplingParams  

llm = LLM(  
    model=model_name,  
    dtype="bfloat16",  
    trust_remote_code=True,  
    max_model_len=1100,
)  

In [ ]:
sampling_params = SamplingParams(  
    max_tokens=200,
    temperature=0.1,
)

In [ ]:
outputs = llm.generate(texts, sampling_params)  

In [ ]:
train_df["explanation"] = [out.outputs[0].text for out in outputs]
train_df.shape

In [ ]:
print(train_df.loc[14100,"explanation"])

In [ ]:
train_df.to_parquet("reasoning-exp04.parquet")

In [ ]:
from datasets import Dataset  
  
dataset = Dataset.from_pandas(train_df)  
  
dataset.push_to_hub(  
    "Norrawee/chess-exp04",  
    private=False  # set True if needed  
)  